# PharmaOps-AI

# Feature Engineering

## Objective

The objective of this notebook is to engineer business-oriented analytical features from the processed pharmacy datasets. These engineered features will support executive reporting, business analytics, Power BI dashboards, and AI-driven recommendation systems.

In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
medicines_df = pd.read_csv("../../data/processed/python/Medicines.csv")

inventory_df = pd.read_csv("../../data/processed/python/Inventory.csv")

suppliers_df = pd.read_csv("../../data/processed/python/Suppliers.csv")

sales_df = pd.read_csv("../../data/processed/python/Sales.csv")

waste_df = pd.read_csv("../../data/processed/python/Waste.csv")

category_df = pd.read_csv("../../data/processed/python/Category.csv")

print("Processed datasets loaded successfully.")

Processed datasets loaded successfully.


In [3]:
inventory_ids = set(inventory_df["Medicine_ID"].astype(str))
sales_ids = set(sales_df["Medicine_ID"].astype(str))

print("Inventory IDs :", len(inventory_ids))
print("Sales IDs     :", len(sales_ids))
print("Matching IDs  :", len(inventory_ids & sales_ids))

print("\nInventory IDs not in Sales (first 20):")
print(sorted(list(inventory_ids - sales_ids))[:20])

print("\nSales IDs not in Inventory (first 20):")
print(sorted(list(sales_ids - inventory_ids))[:20])

Inventory IDs : 4824
Sales IDs     : 4192
Matching IDs  : 4192

Inventory IDs not in Sales (first 20):
['MED000020', 'MED000551', 'MED000613', 'MED000787', 'MED000805', 'MED000911', 'MED001089', 'MED001474', 'MED001626', 'MED001704', 'MED001842', 'MED002160', 'MED002390', 'MED003720', 'MED004956', 'MED005012', 'MED005108', 'MED005165', 'MED005209', 'MED005536']

Sales IDs not in Inventory (first 20):
[]


In [39]:
print("=" * 60)
print("SALES COVERAGE")
print("=" * 60)

print("Medicines with Sales :", (feature_store["Total_Revenue"] > 0).sum())
print("Medicines without Sales :", (feature_store["Total_Revenue"] == 0).sum())

print("\nCoverage %:",
      round(
          (feature_store["Total_Revenue"] > 0).mean() * 100,
          2
      ),
      "%"
)

SALES COVERAGE
Medicines with Sales : 4306
Medicines without Sales : 648

Coverage %: 86.92 %


In [4]:
feature_store = (

    inventory_df

    .merge(
        medicines_df,
        on="Medicine_ID",
        how="left"
    )

    .merge(
        suppliers_df,
        on="Supplier_ID",
        how="left"
    )

    .merge(
        category_df,
        on="Category_ID",
        how="left"
    )

)

sales_summary = (

    sales_df

    .groupby("Medicine_ID")

    .agg(

        Total_Revenue=("Total_Amount","sum"),

        Total_Quantity_Sold=("Quantity_Sold","sum"),

        Average_Discount=("Discount_Percentage","mean"),

        Total_Transactions=("Transaction_ID","count")

    )

    .reset_index()

)

feature_store = feature_store.merge(

    sales_summary,

    on="Medicine_ID",

    how="left"

)

waste_summary = (

    waste_df

    .groupby("Medicine_ID")

    .agg(

        Total_Waste_Value=("Total_Waste_Value","sum"),

        Total_Quantity_Wasted=("Quantity_Wasted","sum")

    )

    .reset_index()

)

feature_store = feature_store.merge(

    waste_summary,

    on="Medicine_ID",

    how="left"

)

feature_store.fillna(0,inplace=True)

print(feature_store.shape)

feature_store.head()

(4954, 46)


,Inventory_ID,Medicine_ID,Supplier_ID,Batch_Number,Manufacturing_Date,Expiry_Date,Quantity_In_Stock,Unit_Cost,Selling_Price,Reorder_Level,Storage_Temperature,Warehouse_Location,Stock_Status,Last_Restock_Date,Product_NDC,Generic_Name,Brand_Name,Manufacturer,Active_Ingredient,Strength,Dosage_Form,Route,Pharm_Class,Category_ID,Supplier_Name,Supplier_Type,Supplier_Category,Preferred_Supplier,City,State,Contact_Email,Contact_Number,Lead_Time_Days,Supplier_Rating,Active_Status,Contract_Start_Date,Contract_End_Date,GST_Number,Category_Name,Description,Total_Revenue,Total_Quantity_Sold,Average_Discount,Total_Transactions,Total_Waste_Value,Total_Quantity_Wasted
0,INV000001,MED103673,SUP0015,BAT126225,2025-02-03,2026-08-27,137,407.64,561.91,51,Room Temperature,WH-E,In Stock,2025-04-16,53747-352,Doxazosin Mesylate,Doxazosin Mesylate,"Unichem Laboratories Limited, India",DOXAZOSIN MESYLATE,2 mg/1,TABLET,ORAL,Adrenergic alpha-Antagonists [MoA],CAT003,Bayer Pharmaceuticals Pvt Ltd,Manufacturer,Domestic,No,Mumbai,Maharashtra,procurement@bayer.com,9212530587,12,4.2,Active,2022-11-19,2025-11-18,27YCEPR2182T1Z5,Diabetes,Medicines used for diabetes management,5587.90,5.0,0.000000,1.0,51362.64,126.0
1,INV000002,MED067629,SUP0005,BAT131244,2024-01-28,2025-07-21,516,20.79,25.77,179,Frozen (-20Â°C),WH-E,In Stock,2024-04-11,72147-1802,Aluminum Zirconium Tetrachlorohydrex Gly,Helmm Antiperspirant Deodorant Night Market,"Zgtl, Llc",ALUMINUM ZIRCONIUM TETRACHLOROHYDREX GLY,18 g/57g,GEL,TOPICAL,UNKNOWN,CAT015,Aurobindo Pharma Ltd,Manufacturer,Domestic,Yes,Hyderabad,Telangana,procurement@aurobindo.com,6943239974,17,4.5,Active,2023-10-09,2027-10-08,36GFYWI6394I1Z5,Others,Medicines that do not fit into the predefined ...,219.29,1.0,5.000000,1.0,3056.13,147.0
2,INV000003,MED067313,SUP0029,BAT571029,2025-02-09,2026-02-04,711,207.45,255.39,133,Refrigerated (2â€“8Â°C),WH-A,In Stock,2025-04-05,0409-2720,Heparin Sodium,Heparin Sodium,"Hospira, Inc.",HEPARIN SODIUM,1000 [USP'U]/mL,"INJECTION, SOLUTION",INTRAVENOUS,Anti-coagulant [EPC],CAT003,Ajanta Pharma Ltd,Manufacturer,Domestic,Yes,Mumbai,Maharashtra,procurement@ajantapharma.com,7191043170,19,4.8,Active,2023-08-09,2026-08-08,27HIODG6704R1Z5,Diabetes,Medicines used for diabetes management,1961.05,3.0,17.500000,2.0,20330.10,98.0
3,INV000004,MED014580,SUP0049,BAT201414,2025-07-25,2027-06-15,267,439.19,574.13,36,Refrigerated (2â€“8Â°C),WH-A,In Stock,2026-01-10,80425-0115,Eszopiclone,Eszopiclone,"Advanced Rx Pharmacy Of Tennessee, Llc",ESZOPICLONE,3 mg/1,"TABLET, FILM COATED",ORAL,UNKNOWN,CAT015,Shree Medical Agencies,Wholesaler,Domestic,No,Ahmedabad,Gujarat,procurement@shreemedicalagencies.in,8140506492,7,3.8,Active,2024-08-25,2027-08-25,24LDEPT1078K1Z5,Others,Medicines that do not fit into the predefined ...,1599.00,3.0,17.500000,2.0,74662.30,170.0
4,INV000005,MED089590,SUP0038,BAT969693,2025-07-30,2028-01-16,713,32.27,40.19,42,Refrigerated (2â€“8Â°C),WH-A,In Stock,2025-10-02,0220-2334,Grindelia Hirsutula Flowering Top,Grindelia,Boiron,GRINDELIA HIRSUTULA FLOWERING TOP,200 [kp_C]/200[kp_C],PELLET,ORAL,UNKNOWN,CAT015,CarePlus Distributors,Distributor,Domestic,No,Bengaluru,Karnataka,procurement@careplusdistributors.in,7032183433,10,4.4,Active,2023-03-30,2027-03-29,29RHPMN4737I1Z5,Others,Medicines that do not fit into the predefined ...,3353.92,8.0,8.333333,3.0,4517.80,140.0


In [7]:
inventory_ids = set(inventory_df["Medicine_ID"].astype(str))
sales_ids = set(sales_df["Medicine_ID"].astype(str))

print("=" * 60)
print("SALES DATA VALIDATION")
print("=" * 60)

print("Unique Inventory Medicines :", len(inventory_ids))
print("Unique Sales Medicines     :", len(sales_ids))
print("Matching Medicines         :", len(inventory_ids & sales_ids))

SALES DATA VALIDATION
Unique Inventory Medicines : 4824
Unique Sales Medicines     : 4192
Matching Medicines         : 4192


In [8]:
date_columns = [

    "Manufacturing_Date",

    "Expiry_Date",

    "Last_Restock_Date",

    "Contract_Start_Date",

    "Contract_End_Date"

]

for col in date_columns:

    feature_store[col] = pd.to_datetime(feature_store[col])

print("Date conversion completed.")

Date conversion completed.


In [9]:
print("="*70)
print("Feature Store Summary")
print("="*70)

print(f"Rows    : {feature_store.shape[0]}")
print(f"Columns : {feature_store.shape[1]}")

print()

print(feature_store.info())

Feature Store Summary
Rows    : 4954
Columns : 46

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4954 entries, 0 to 4953
Data columns (total 46 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Inventory_ID           4954 non-null   object        
 1   Medicine_ID            4954 non-null   object        
 2   Supplier_ID            4954 non-null   object        
 3   Batch_Number           4954 non-null   object        
 4   Manufacturing_Date     4954 non-null   datetime64[ns]
 5   Expiry_Date            4954 non-null   datetime64[ns]
 6   Quantity_In_Stock      4954 non-null   int64         
 7   Unit_Cost              4954 non-null   float64       
 8   Selling_Price          4954 non-null   float64       
 9   Reorder_Level          4954 non-null   int64         
 10  Storage_Temperature    4954 non-null   object        
 11  Warehouse_Location     4954 non-null   object        
 12  Stock_Statu

# Financial Feature Engineering

This section derives financial metrics that measure inventory investment, profitability, revenue generation, and business performance. These features will support executive dashboards and profitability analysis.

In [10]:
# Total Inventory Investment

feature_store["Inventory_Value"] = (
    feature_store["Quantity_In_Stock"] *
    feature_store["Unit_Cost"]
)

# Potential Sales Value

feature_store["Potential_Sales_Value"] = (
    feature_store["Quantity_In_Stock"] *
    feature_store["Selling_Price"]
)

# Gross Profit based on Potential Sales and Inventory Cost
feature_store["Potential_Gross_Profit"] = (
    feature_store["Potential_Sales_Value"] -
    feature_store["Inventory_Value"]
)

# Prevent negative profit values
feature_store["Potential_Gross_Profit"] = (
    feature_store["Potential_Gross_Profit"]
    .clip(lower=0)
)
print("Inventory financial features created.")

Inventory financial features created.


In [11]:
# Profit per Unit
feature_store["Profit_per_Unit"] = (
    feature_store["Selling_Price"] -
    feature_store["Unit_Cost"]
)

# Profit Margin Percentage
feature_store["Profit_Margin_Percentage"] = (
    feature_store["Potential_Gross_Profit"] /
    feature_store["Potential_Sales_Value"]
) * 100

feature_store["Profit_Margin_Percentage"] = (
    feature_store["Profit_Margin_Percentage"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

# Markup Percentage
feature_store["Markup_Percentage"] = (
    feature_store["Profit_per_Unit"] /
    feature_store["Unit_Cost"]
) * 100

feature_store["Markup_Percentage"] = (
    feature_store["Markup_Percentage"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)
print("Profitability features created.")

Profitability features created.


In [12]:
feature_store["Revenue_per_Transaction"] = (
    feature_store["Total_Revenue"] /
    feature_store["Total_Transactions"]
)

feature_store["Revenue_per_Unit_Sold"] = (
    feature_store["Total_Revenue"] /
    feature_store["Total_Quantity_Sold"]
)

feature_store.fillna(0, inplace=True)

print("Revenue performance features created.")

Revenue performance features created.


In [13]:
# Inventory ROI
feature_store["Inventory_ROI"] = (
    feature_store["Potential_Gross_Profit"] /
    feature_store["Inventory_Value"]
) * 100

feature_store["Inventory_ROI"] = (
    feature_store["Inventory_ROI"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

# Cost to Revenue Ratio
feature_store["Cost_to_Revenue_Ratio"] = (
    feature_store["Inventory_Value"] /
    feature_store["Potential_Sales_Value"]
) * 100

feature_store["Cost_to_Revenue_Ratio"] = (
    feature_store["Cost_to_Revenue_Ratio"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

print("ROI features created.")

ROI features created.


In [14]:
financial_features = [

    "Inventory_Value",

    "Potential_Sales_Value",

    "Potential_Gross_Profit",

    "Profit_per_Unit",

    "Profit_Margin_Percentage",

    "Markup_Percentage",

    "Revenue_per_Transaction",

    "Revenue_per_Unit_Sold",

    "Inventory_ROI",

    "Cost_to_Revenue_Ratio"

]

feature_store[
    financial_features
].describe().T

,count,mean,std,min,25%,50%,75%,max
Inventory_Value,4954.0,147991.078837,129049.924637,0.00,40794.555000,111849.690000,228258.957500,592145.760000
Potential_Sales_Value,4954.0,192186.221591,167834.261712,0.00,53081.070000,145264.480000,296697.847500,795485.080000
Potential_Gross_Profit,4954.0,44195.142753,40036.430525,0.00,11808.670000,32387.250000,67559.662500,221129.920000
Profit_per_Unit,4954.0,89.568932,55.289141,1.06,43.747500,85.185000,129.647500,237.590000
Profit_Margin_Percentage,4954.0,22.804542,3.699273,0.00,19.896708,23.057173,25.913452,28.566130
Markup_Percentage,4954.0,29.927155,5.783718,20.00,24.865953,29.983144,34.981985,39.989616
Revenue_per_Transaction,4954.0,1221.870595,1114.197355,0.00,485.523333,984.225000,1673.031875,10133.800000
Revenue_per_Unit_Sold,4954.0,503.915816,310.587983,0.00,281.597500,522.410000,738.485313,1198.850000
Inventory_ROI,4954.0,29.829823,6.045005,0.00,24.838815,29.966631,34.977271,39.989616
Cost_to_Revenue_Ratio,4954.0,76.832115,5.772271,0.00,74.055860,76.904755,80.070711,83.333333


# Inventory Feature Engineering

This section develops inventory intelligence features that measure stock health, inventory ageing, expiry risk, replenishment requirements, and inventory criticality. These engineered features support inventory optimization and operational decision-making.

In [15]:
from datetime import datetime

today = pd.Timestamp.today().normalize()

feature_store["Days_to_Expiry"] = (
    feature_store["Expiry_Date"] - today
).dt.days

feature_store["Inventory_Age_Days"] = (
    today - feature_store["Manufacturing_Date"]
).dt.days

feature_store["Days_Since_Last_Restock"] = (
    today - feature_store["Last_Restock_Date"]
).dt.days

print("Date-based inventory features created.")

Date-based inventory features created.


In [16]:
feature_store["Stock_Difference"] = (
    feature_store["Quantity_In_Stock"] -
    feature_store["Reorder_Level"]
)

feature_store["Stock_Coverage_Ratio"] = (
    feature_store["Quantity_In_Stock"] /
    feature_store["Reorder_Level"]
)

feature_store["Stock_Coverage_Ratio"] = (
    feature_store["Stock_Coverage_Ratio"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

print("Inventory risk features created.")

Inventory risk features created.


In [17]:
feature_store["Low_Stock_Flag"] = np.where(
    feature_store["Quantity_In_Stock"] <= feature_store["Reorder_Level"],
    1,
    0
)

feature_store["Overstock_Flag"] = np.where(
    feature_store["Quantity_In_Stock"] >=
    feature_store["Reorder_Level"] * 3,
    1,
    0
)

feature_store["Out_of_Stock_Flag"] = np.where(
    feature_store["Quantity_In_Stock"] == 0,
    1,
    0
)

print("Inventory classification features created.")

Inventory classification features created.


In [18]:
feature_store["Expiry_Risk"] = pd.cut(

    feature_store["Days_to_Expiry"],

    bins=[
        -9999,
        0,
        30,
        90,
        180,
        99999
    ],

    labels=[
        "Expired",
        "Critical",
        "High",
        "Medium",
        "Low"
    ]

)

feature_store["Near_Expiry_Flag"] = np.where(
    feature_store["Days_to_Expiry"] <= 90,
    1,
    0
)

print("Expiry features created.")

Expiry features created.


In [19]:
score = np.where(
    feature_store["Low_Stock_Flag"] == 1,
    -20,
    0
)

score += np.where(
    feature_store["Overstock_Flag"] == 1,
    -15,
    0
)

score += np.where(
    feature_store["Near_Expiry_Flag"] == 1,
    -25,
    0
)

score += np.where(
    feature_store["Supplier_Rating"] >= 4.5,
    15,
    5
)

feature_store["Inventory_Health_Score"] = (
    100 + score
)

feature_store["Inventory_Health_Score"] = (
    feature_store["Inventory_Health_Score"]
    .clip(0,100)
)

print("Inventory Health Score created.")

Inventory Health Score created.


In [20]:
feature_store["Business_Priority"] = np.select(

    [

        (
            (feature_store["Inventory_Health_Score"] < 50)
        ),

        (
            (feature_store["Inventory_Health_Score"] >= 50)
            &
            (feature_store["Inventory_Health_Score"] < 75)
        ),

        (
            feature_store["Inventory_Health_Score"] >= 75
        )

    ],

    [

        "High",

        "Medium",

        "Low"

    ],

    default="Low"

)

print("Business Priority created.")

Business Priority created.


In [21]:
inventory_features = [

    "Days_to_Expiry",

    "Inventory_Age_Days",

    "Days_Since_Last_Restock",

    "Stock_Difference",

    "Stock_Coverage_Ratio",

    "Low_Stock_Flag",

    "Overstock_Flag",

    "Out_of_Stock_Flag",

    "Near_Expiry_Flag",

    "Inventory_Health_Score",

    "Business_Priority"

]

feature_store[
    inventory_features
].head()

,Days_to_Expiry,Inventory_Age_Days,Days_Since_Last_Restock,Stock_Difference,Stock_Coverage_Ratio,Low_Stock_Flag,Overstock_Flag,Out_of_Stock_Flag,Near_Expiry_Flag,Inventory_Health_Score,Business_Priority
0,20,550,478,86,2.686275,0,0,0,1,80,Low
1,-382,922,848,337,2.882682,0,0,0,1,90,Low
2,-184,544,489,578,5.345865,0,1,0,1,75,Low
3,312,378,209,231,7.416667,0,1,0,0,90,Low
4,527,373,309,671,16.976190,0,1,0,0,90,Low


# Advanced Inventory Classification

This section applies industry-standard inventory management techniques such as ABC Analysis, Inventory Criticality, and Stock Priority Classification to support procurement and inventory optimization.

In [22]:
feature_store = feature_store.sort_values(
    by="Inventory_Value",
    ascending=False
)

feature_store["Cumulative_Inventory_Value"] = (
    feature_store["Inventory_Value"].cumsum()
)

total_inventory_value = feature_store["Inventory_Value"].sum()

feature_store["Cumulative_Percentage"] = (
    feature_store["Cumulative_Inventory_Value"]
    / total_inventory_value
) * 100

feature_store["ABC_Class"] = np.select(

    [

        feature_store["Cumulative_Percentage"] <= 80,

        feature_store["Cumulative_Percentage"] <= 95

    ],

    [

        "A",

        "B"

    ],

    default="C"

)

print("ABC Classification created.")

ABC Classification created.


In [23]:
feature_store["Inventory_Criticality"] = np.select(

    [

        (
            (feature_store["ABC_Class"]=="A")
            &
            (feature_store["Near_Expiry_Flag"]==1)
        ),

        (
            feature_store["ABC_Class"]=="A"
        ),

        (
            feature_store["ABC_Class"]=="B"
        )

    ],

    [

        "Critical",

        "High",

        "Medium"

    ],

    default="Low"

)

print("Inventory Criticality created.")

Inventory Criticality created.


In [24]:
feature_store["Stock_Priority"] = np.select(

    [

        feature_store["Low_Stock_Flag"]==1,

        feature_store["Overstock_Flag"]==1,

        feature_store["Near_Expiry_Flag"]==1

    ],

    [

        "Immediate Replenishment",

        "Reduce Inventory",

        "Sell Immediately"

    ],

    default="Healthy Stock"

)

print("Executive stock priority created.")

Executive stock priority created.


# Sales Feature Engineering

This section derives business features that measure sales performance, medicine movement, customer demand, pricing effectiveness, and revenue contribution.

In [25]:
feature_store["Revenue_Percentile"] = (
    feature_store["Total_Revenue"]
    .rank(method="average", pct=True)
)

feature_store["Revenue_Category"] = np.select(

    [
        feature_store["Revenue_Percentile"] <= 0.25,
        feature_store["Revenue_Percentile"] <= 0.50,
        feature_store["Revenue_Percentile"] <= 0.75
    ],

    [
        "Low",
        "Medium",
        "High"
    ],

    default="Very High"

)

print("Revenue Category created.")

Revenue Category created.


In [26]:
feature_store["Sales_Percentile"] = (
    feature_store["Total_Quantity_Sold"]
    .rank(method="average", pct=True)
)

feature_store["Movement_Type"] = np.select(

    [
        feature_store["Sales_Percentile"] <= 0.33,
        feature_store["Sales_Percentile"] <= 0.66
    ],

    [
        "Slow Moving",
        "Medium Moving"
    ],

    default="Fast Moving"

)

print("Movement Type created.")

Movement Type created.


In [27]:
threshold = feature_store["Total_Revenue"].quantile(0.80)

feature_store["High_Revenue_Flag"] = np.where(

    feature_store["Total_Revenue"] >= threshold,

    1,

    0

)

print("High Revenue Flag created.")

High Revenue Flag created.


In [28]:
feature_store["Sales_Performance_Score"] = (

    feature_store["Total_Revenue"].rank(pct=True)*40 +

    feature_store["Total_Quantity_Sold"].rank(pct=True)*30 +

    feature_store["Profit_Margin_Percentage"].rank(pct=True)*30

).round(2)

print("Sales Performance Score created.")

Sales Performance Score created.


In [29]:
feature_store["Sales_Performance_Score"] = (

    feature_store["Total_Revenue"].rank(pct=True)*40 +

    feature_store["Total_Quantity_Sold"].rank(pct=True)*30 +

    feature_store["Profit_Margin_Percentage"].rank(pct=True)*30

).round(2)

print("Sales Performance Score created.")

Sales Performance Score created.


In [30]:
feature_store["Waste_Cost_Ratio"] = (

    feature_store["Total_Waste_Value"] /

    feature_store["Inventory_Value"]

)

feature_store["Waste_Cost_Ratio"] = (

    feature_store["Waste_Cost_Ratio"]

    .replace([np.inf,-np.inf],0)

    .fillna(0)

)

print("Waste Cost Ratio created.")

Waste Cost Ratio created.


In [31]:
# =====================================================
# WASTE FEATURES
# =====================================================

# Waste Percentage
feature_store["Waste_Percentage"] = (
    feature_store["Total_Waste_Value"] /
    feature_store["Inventory_Value"]
) * 100

feature_store["Waste_Percentage"] = (
    feature_store["Waste_Percentage"]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

print("Waste Percentage created.")

# Waste Risk
feature_store["Waste_Risk"] = np.select(

    [

        feature_store["Waste_Percentage"] <= 5,

        feature_store["Waste_Percentage"] <= 15,

        feature_store["Waste_Percentage"] <= 30

    ],

    [

        "Low",

        "Medium",

        "High"

    ],

    default="Critical"

)

print("Waste Risk created.")

Waste Percentage created.
Waste Risk created.


In [32]:
feature_store["Supplier_Performance_Index"] = (

    feature_store["Supplier_Rating"]*20 -

    feature_store["Lead_Time_Days"]

).round(2)

print("Supplier Performance Index created.")

Supplier Performance Index created.


In [33]:
feature_store["Supplier_Risk"] = np.select(

    [

        feature_store["Lead_Time_Days"] >= 15,

        feature_store["Lead_Time_Days"] >= 10

    ],

    [

        "High",

        "Medium"

    ],

    default="Low"

)

print("Supplier Risk created.")

Supplier Risk created.


In [34]:
feature_store["Executive_Risk_Score"] = (

    feature_store["Inventory_Health_Score"]*0.35 +

    feature_store["Sales_Performance_Score"]*0.35 +

    feature_store["Supplier_Performance_Index"]*0.15 +

    (100-feature_store["Waste_Percentage"])*0.15

).round(2)

print("Executive Risk Score created.")

Executive Risk Score created.


In [35]:
feature_store["Executive_Priority"] = np.select(

    [

        feature_store["Executive_Risk_Score"] < 40,

        feature_store["Executive_Risk_Score"] < 60,

        feature_store["Executive_Risk_Score"] < 80

    ],

    [

        "Critical",

        "High",

        "Medium"

    ],

    default="Low"

)

print("Executive Priority created.")

Executive Priority created.


In [36]:
print("="*80)
print("FEATURE STORE SUMMARY")
print("="*80)

print(f"Rows    : {feature_store.shape[0]}")
print(f"Columns : {feature_store.shape[1]}")

print()

print("Engineered Features")

engineered = [

    c

    for c in feature_store.columns

    if c not in inventory_df.columns

]

print(len(engineered))

feature_store.head()

FEATURE STORE SUMMARY
Rows    : 4954
Columns : 86

Engineered Features
72


,Inventory_ID,Medicine_ID,Supplier_ID,Batch_Number,Manufacturing_Date,Expiry_Date,Quantity_In_Stock,Unit_Cost,Selling_Price,Reorder_Level,Storage_Temperature,Warehouse_Location,Stock_Status,Last_Restock_Date,Product_NDC,Generic_Name,Brand_Name,Manufacturer,Active_Ingredient,Strength,Dosage_Form,Route,Pharm_Class,Category_ID,Supplier_Name,Supplier_Type,Supplier_Category,Preferred_Supplier,City,State,Contact_Email,Contact_Number,Lead_Time_Days,Supplier_Rating,Active_Status,Contract_Start_Date,Contract_End_Date,GST_Number,Category_Name,Description,Total_Revenue,Total_Quantity_Sold,Average_Discount,Total_Transactions,Total_Waste_Value,Total_Quantity_Wasted,Inventory_Value,Potential_Sales_Value,Potential_Gross_Profit,Profit_per_Unit,Profit_Margin_Percentage,Markup_Percentage,Revenue_per_Transaction,Revenue_per_Unit_Sold,Inventory_ROI,Cost_to_Revenue_Ratio,Days_to_Expiry,Inventory_Age_Days,Days_Since_Last_Restock,Stock_Difference,Stock_Coverage_Ratio,Low_Stock_Flag,Overstock_Flag,Out_of_Stock_Flag,Expiry_Risk,Near_Expiry_Flag,Inventory_Health_Score,Business_Priority,Cumulative_Inventory_Value,Cumulative_Percentage,ABC_Class,Inventory_Criticality,Stock_Priority,Revenue_Percentile,Revenue_Category,Sales_Percentile,Movement_Type,High_Revenue_Flag,Sales_Performance_Score,Waste_Cost_Ratio,Waste_Percentage,Waste_Risk,Supplier_Performance_Index,Supplier_Risk,Executive_Risk_Score,Executive_Priority
2338,INV002361,MED000911,SUP0048,BAT820020,2026-03-06,2027-03-31,993,596.32,798.36,87,Refrigerated (2â€“8Â°C),WH-A,In Stock,2026-05-07,76420-836,Gabapentin,Gabapentin,"Asclemed Usa, Inc.",GABAPENTIN,600 mg/1,TABLET,ORAL,Decreased Central Nervous System Disorganized ...,CAT005,Universal Pharma Traders,Wholesaler,Domestic,Yes,Hyderabad,Telangana,procurement@universaltraders.in,6562697526,4,4.0,Inactive,2025-05-31,2028-05-30,36SIAKF8196E1Z5,Neurology,Medicines for neurological disorders and the c...,0.00,0.0,0.0,0.0,0.00,0.0,592145.76,792771.48,200625.72,202.04,25.306879,33.881138,0.00,0.000000,33.881138,74.693121,236,154,92,906,11.413793,0,1,0,Low,0,90,Low,592145.76,0.080768,A,High,Reduce Inventory,0.065503,Low,0.065503,Slow Moving,0,25.40,0.000000,0.000000,Low,76.0,Low,66.79,Medium
4198,INV004233,MED056437,SUP0058,BAT986959,2026-03-05,2027-03-30,986,597.63,806.78,118,Frozen (-20Â°C),WH-A,In Stock,2026-04-23,98132-316,Titanium Dioxide,Bareminerals Barepro Performance Wear Liquid F...,Orveon Global Us Llc,TITANIUM DIOXIDE,33.2 mg/mL,LIQUID,TOPICAL,UNKNOWN,CAT015,RK Pharma Agencies,Wholesaler,Domestic,No,Visakhapatnam,Andhra Pradesh,procurement@rkagencies.in,8489250532,4,4.3,Active,2022-12-26,2025-12-25,37DOZNX8284Z1Z5,Others,Medicines that do not fit into the predefined ...,1593.69,6.0,5.0,3.0,2390.52,4.0,589263.18,795485.08,206221.90,209.15,25.924044,34.996570,531.23,265.615000,34.996570,74.075956,235,155,106,868,8.355932,0,1,0,Low,0,90,Low,1181408.94,0.161142,A,High,Reduce Inventory,0.432983,Medium,0.656944,Medium Moving,0,59.56,0.004057,0.405680,Low,82.0,Low,79.59,Medium
3395,INV003424,MED000320,SUP0031,BAT669736,2026-01-22,2028-05-11,965,596.60,723.41,192,Frozen (-20Â°C),WH-E,In Stock,2026-03-21,0527-1927,Methadone Hydrochloride,"Methadone Hydrocloride Dye-Free, Sugar-Free, U...","Lannett Company, Inc.",METHADONE HYDROCHLORIDE,10 mg/mL,CONCENTRATE,ORAL,Full Opioid Agonists [MoA],CAT004,MedPlus Health Services,Distributor,Domestic,No,Hyderabad,Telangana,procurement@medplusmart.com,9155229266,6,4.3,Active,2025-10-01,2030-09-30,36USVXZ5956C1Z5,Pain Management,Pain relief and anti-inflammatory medicines,629.68,1.0,15.0,1.0,4772.80,8.0,575719.00,698090.65,122371.65,126.81,17.529478,21.255448,629.68,629.680000,21.255448,82.470522,643,197,139,773,5.026042,0,1,0,Low,0,90,Low,1757127.94,0.239669,A,High,Reduce Inventory,0.225474,Low,0.186617,Slow Moving,0,16.67,0.008290,0.829016,Low,80.0,Low,64.21,Medium
749,INV000760,MED096650,SUP0014,BAT840719,2023-11-01,2025-12-20,979,584.92,793.34,53,Frozen (-20Â°C),WH-A,In Stock,2024-03-12,82969-2192,"Berberis Vulg, 

In [37]:
feature_store.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Inventory_ID,4954,4954,INV002163,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Medicine_ID,4954,4824,MED134145,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Supplier_ID,4954,75,SUP0040,87,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Batch_Number,4954,4954,BAT656999,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Manufacturing_Date,4954,NaN,NaN,NaN,2025-01-20 00:45:03.270084864,2023-07-21 00:00:00,2024-04-13 00:00:00,2025-01-26 00:00:00,2025-10-26 00:00:00,2026-07-20 00:00:00,NaN
...,...,...,...,...,...,...,...,...,...,...,...
Waste_Risk,4954,4,Low,2721,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Supplier_Performance_Index,4954.0,NaN,NaN,NaN,78.38696,63.0,72.0,78.0,85.0,92.0,7.315607
Supplier_Risk,4954,3,Low,3192,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Executive_Risk_Score,4954.0,NaN,NaN,NaN,71.63082,-30.43,65.2125,71.88,78.6875,97.94,10.087152


In [38]:
import os

os.makedirs("../../outputs/processed_data", exist_ok=True)

feature_store.to_csv(

    "../../outputs/processed_data/Feature_Store.csv",

    index=False

)

print("="*60)
print("Feature Store exported successfully.")
print("="*60)

Feature Store exported successfully.
